# 11 — Hospital Synthetic Dataset 3,000 Rows

Tạo dữ liệu mô phỏng từ Cleveland cho hai mục đích: `enriched_train_3000.csv` để thử nghiệm ML và `hospital_demo_synthetic_3000.csv` để demo/ops. Cỡ mẫu độc lập thực tế vẫn là 303 bệnh nhân; synthetic rows không được xem là bệnh nhân thật hoặc external validation.

In [ ]:
!pip install -q gdown sdv

In [ ]:
from pathlib import Path
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ks_2samp
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TARGET_ROWS = 3000
OUTPUT_DIR = Path('/content/hospital_data_3000')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(RANDOM_STATE)

## 1. Load dữ liệu và khóa test thật

Generator chỉ học khoảng 242 dòng train. Test thật 20% không tham gia fit hoặc quality tuning.

In [ ]:
FILE_ID = '1YTzUy_RreXqnM5fqMR0hOLZPeXOvYoju'
DATA_PATH = Path('/content/cleveland.csv')
gdown.download(id=FILE_ID, output=str(DATA_PATH), quiet=False)

FEATURES = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
            'exang','oldpeak','slope','ca','thal']
NUMERICAL_FEATURES = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL_FEATURES = [c for c in FEATURES if c not in NUMERICAL_FEATURES]
ALL_COLUMNS = FEATURES + ['target']

df = pd.read_csv(DATA_PATH, header=None, names=FEATURES+['target_original'], na_values=['?'])
df['target'] = (pd.to_numeric(df['target_original'], errors='coerce') > 0).astype(int)
for column in FEATURES:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df = df[ALL_COLUMNS].drop_duplicates().reset_index(drop=True)
train_df, locked_test_df = train_test_split(
    df, test_size=0.20, stratify=df['target'], random_state=RANDOM_STATE)
train_df = train_df.reset_index(drop=True)
locked_test_df = locked_test_df.reset_index(drop=True)
assert len(df) == 303 and len(train_df) + len(locked_test_df) == 303
print('Original:', df.shape, 'Generator train:', train_df.shape, 'Locked test:', locked_test_df.shape)
display(train_df['target'].value_counts(normalize=True).rename('train_rate'))

## 2. Fit Gaussian Copula theo từng target class

Fit riêng target 0/1 giúp giữ tỷ lệ bệnh và tránh bộ 3.000 dòng bị lệch lớp do sampling. Categorical values được đưa về mã hợp lệ sau khi sinh.

In [ ]:
CLINICAL_BOUNDS = {
    'age':(18,100), 'trestbps':(60,260), 'chol':(80,800),
    'thalach':(40,240), 'oldpeak':(0.0,10.0)}
INTEGER_NUMERICAL = ['age','trestbps','chol','thalach']

def make_metadata(frame):
    metadata = Metadata.detect_from_dataframe(data=frame, table_name='patients')
    for column in CATEGORICAL_FEATURES:
        metadata.update_column(column_name=column, sdtype='categorical')
    for column in NUMERICAL_FEATURES:
        metadata.update_column(column_name=column, sdtype='numerical')
    metadata.validate()
    return metadata

def nearest_allowed(series, allowed):
    allowed = np.asarray(sorted(allowed), dtype=float)
    values = pd.to_numeric(series, errors='coerce').to_numpy(dtype=float)
    valid = ~np.isnan(values)
    values[valid] = allowed[np.abs(values[valid,None]-allowed[None,:]).argmin(axis=1)]
    return values

def enforce_domains(sample, reference):
    result = sample.copy()
    for column, (low, high) in CLINICAL_BOUNDS.items():
        result[column] = pd.to_numeric(result[column], errors='coerce').clip(low, high)
    result[INTEGER_NUMERICAL] = result[INTEGER_NUMERICAL].round()
    result['oldpeak'] = result['oldpeak'].round(1)
    for column in CATEGORICAL_FEATURES:
        allowed = reference[column].dropna().unique()
        result[column] = nearest_allowed(result[column], allowed)
    return result[FEATURES]

synthesizers = {}
for label, class_data in train_df.groupby('target'):
    reference = class_data[FEATURES].reset_index(drop=True)
    model = GaussianCopulaSynthesizer(
        make_metadata(reference), enforce_min_max_values=True, enforce_rounding=True)
    model.fit(reference)
    synthesizers[int(label)] = (model, reference)
print('Fitted class-specific synthesizers:', sorted(synthesizers))

## 3. Tạo enriched train và demo dataset

`enriched_train_3000.csv` chứa 242 train rows thật cộng synthetic. `hospital_demo_synthetic_3000.csv` là synthetic-only, an toàn hơn cho demo hệ thống nhưng vẫn cần kiểm tra privacy/memorization.

In [ ]:
class_rates = train_df['target'].value_counts(normalize=True).sort_index()

def allocate_counts(total):
    count0 = int(round(total * class_rates.loc[0]))
    return {0:count0, 1:total-count0}

def sample_synthetic(total_rows, seed):
    np.random.seed(seed)
    parts = []
    for label, rows in allocate_counts(total_rows).items():
        model, reference = synthesizers[label]
        sampled = enforce_domains(model.sample(num_rows=rows), reference)
        sampled['target'] = label
        sampled['data_origin'] = 'synthetic_gaussian'
        parts.append(sampled)
    return pd.concat(parts, ignore_index=True).sample(
        frac=1, random_state=seed).reset_index(drop=True)

n_enriched_synthetic = TARGET_ROWS - len(train_df)
enriched_synthetic = sample_synthetic(n_enriched_synthetic, RANDOM_STATE)
real_part = train_df.copy(); real_part['data_origin'] = 'real_train'
enriched_train = pd.concat([real_part, enriched_synthetic], ignore_index=True).sample(
    frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
demo_synthetic = sample_synthetic(TARGET_ROWS, RANDOM_STATE + 100)

assert len(enriched_train) == TARGET_ROWS
assert len(demo_synthetic) == TARGET_ROWS
assert set(enriched_train['target'].unique()) == {0,1}
print('Enriched train:', enriched_train.shape)
print('Synthetic demo:', demo_synthetic.shape)
display(pd.DataFrame({
    'real_train':train_df['target'].value_counts(normalize=True),
    'enriched':enriched_train['target'].value_counts(normalize=True),
    'demo':demo_synthetic['target'].value_counts(normalize=True)}).round(4))

## 4. Quality gates trước khi sử dụng

KS/TV càng thấp càng giống. Exact-match phải thấp. Việc vượt quality gate không chứng minh dữ liệu đúng lâm sàng; cần bác sĩ hoặc chuyên gia domain kiểm tra thêm.

In [ ]:
def total_variation(real, synthetic):
    categories = sorted(set(real.dropna().unique()) | set(synthetic.dropna().unique()))
    p = real.value_counts(normalize=True).reindex(categories, fill_value=0)
    q = synthetic.value_counts(normalize=True).reindex(categories, fill_value=0)
    return float(0.5 * np.abs(p-q).sum())

quality_rows = []
for column in NUMERICAL_FEATURES:
    quality_rows.append({'feature':column, 'type':'numeric_KS',
        'distance':ks_2samp(train_df[column].dropna(), demo_synthetic[column].dropna()).statistic})
for column in CATEGORICAL_FEATURES + ['target']:
    quality_rows.append({'feature':column, 'type':'categorical_TV',
        'distance':total_variation(train_df[column], demo_synthetic[column])})
quality = pd.DataFrame(quality_rows).sort_values('distance', ascending=False)
exact_matches = demo_synthetic[ALL_COLUMNS].merge(
    train_df[ALL_COLUMNS].drop_duplicates(), how='inner').shape[0]
exact_match_rate = exact_matches / len(demo_synthetic)
display(quality.round(4))
print('Exact matches:', exact_matches, 'Rate:', round(exact_match_rate, 4))

plt.figure(figsize=(13,5))
sns.barplot(data=quality, x='feature', y='distance', hue='type')
plt.xticks(rotation=45); plt.title('Synthetic quality distances'); plt.show()

## 5. Lưu dataset và manifest

Các file chứa cột `data_origin` để luôn phân biệt real/synthetic. Không dùng `hospital_demo_synthetic_3000.csv` làm test lâm sàng.

In [ ]:
enriched_path = OUTPUT_DIR / 'enriched_train_3000.csv'
demo_path = OUTPUT_DIR / 'hospital_demo_synthetic_3000.csv'
test_path = OUTPUT_DIR / 'locked_real_test.csv'
manifest_path = OUTPUT_DIR / 'README_DATASET.txt'

enriched_train.to_csv(enriched_path, index=False)
demo_synthetic.to_csv(demo_path, index=False)
locked_test_df.to_csv(test_path, index=False)
manifest_path.write_text(
    'SOURCE: Cleveland Heart Disease, 303 real patients\n'
    'GENERATOR FIT: train split only\n'
    'ENRICHED: real train + Gaussian synthetic, total 3000\n'
    'DEMO: 3000 synthetic rows\n'
    'WARNING: synthetic rows are not independent real patients and are not clinical validation data\n',
    encoding='utf-8')

for path in [enriched_path, demo_path, test_path, manifest_path]:
    print(path, '-', path.stat().st_size, 'bytes')

# Optional download in Colab:
# from google.colab import files
# files.download(str(enriched_path))
# files.download(str(demo_path))

## Tài liệu tham khảo và phạm vi giả định

1. SDV Documentation — GaussianCopulaSynthesizer: https://docs.sdv.dev/sdv/single-table-data/modeling/synthesizers/gaussiancopulasynthesizer
2. Xu et al. (2019), *Modeling Tabular Data using Conditional GAN*, NeurIPS: https://papers.neurips.cc/paper/8953-modeling-tabular-data-using-conditional-gan
3. Choi et al. (2017), *Generating Multi-label Discrete Patient Records using GANs*, MLHC: https://proceedings.mlr.press/v68/choi17a.html
4. Yan et al. (2022), *A Multifaceted Benchmarking of Synthetic EHR Generation Models*: https://arxiv.org/abs/2208.01230
5. Hernandez et al. (2023), *Synthetic Tabular Data Evaluation in the Health Domain*: https://pmc.ncbi.nlm.nih.gov/articles/PMC10306449/
6. TRIPOD+AI statement về báo cáo và validation clinical prediction models: https://pmc.ncbi.nlm.nih.gov/articles/PMC11019967/

**Lưu ý:** Các `CLINICAL_BOUNDS` trong notebook là sanity-check kỹ thuật để chặn giá trị mô phỏng cực đoan, được chọn dựa trên phạm vi biến của Cleveland và biên mở rộng. Chúng chưa được xác nhận là clinical guideline. Trước khi dùng với dữ liệu bệnh viện, cần bác sĩ/chuyên gia domain phê duyệt từng giới hạn và quy tắc kết hợp feature.

## 6. Cách sử dụng đúng

- ML experiment: train bằng `enriched_train_3000.csv`, đánh giá bằng `locked_real_test.csv`.
- Demo/ops: dùng `hospital_demo_synthetic_3000.csv`.
- Báo cáo: ghi rõ 3.000 hồ sơ là dữ liệu mô phỏng được sinh từ 303 bệnh nhân Cleveland.
- Không tuyên bố cỡ mẫu nghiên cứu là 3.000 và không dùng synthetic-only test để báo cáo hiệu năng lâm sàng.